In [1]:
import sys; print(sys.executable)
import tensorflow as tf
from dense_pcn_layer import DensePCNLayer
from conv_pcn_layer import Conv2DPCNLayer
from transformer_pcn_layer import TransformerPCNLayer, PositionalEncodingLayer
%load_ext autoreload
%autoreload 2

c:\Users\user\miniconda3\envs\tf_py3.10\python.exe


In [2]:
tf.config.set_visible_devices([], 'GPU')

In [ ]:
from urllib.request import urlretrieve
urlretrieve("http://images.cocodataset.org/zips/train2017.zip", "train2017.zip")
urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "coco.zip")

('coco.zip', <http.client.HTTPMessage at 0x2b788ba3520>)

In [ ]:
import zipfile
zipfile.ZipFile('train2017.zip', 'r').extractall('train2017')
zipfile.ZipFile('coco.zip', 'r').extractall('coco')

In [3]:
import json
train_captions = json.load(open("coco/annotations/captions_train2017.json", 'r'))

In [4]:
import cv2
image_data = []
image_data_id = []
for image in train_captions['images'][:1000]:
    img = cv2.imread(f"train2017/train2017/{image['file_name']}")
    img = cv2.resize(img, (572, 572))
    image_data.append(img)
    image_data_id.append(image["id"])

In [5]:
import numpy as np
image_data = np.array(image_data)
image_data_id = np.array(image_data_id)

In [6]:
caption_data = []
caption_data_image_id =[]
for annotation in train_captions['annotations']:
    if (image_data_id == annotation['image_id']).sum() == 1:
        caption_data.append(annotation['caption'])
        caption_data_image_id.append(annotation['image_id'])
caption_data = np.array(caption_data)
caption_data_image_id = np.array(caption_data_image_id)

In [7]:
chars = set()
for caption in caption_data:
    for char in caption:
        chars.add(char)

In [8]:
chars.add('\0')
chars_arr = np.array(list(chars))

In [9]:
num_tokens = np.array([len(caption) for caption in caption_data]).max()

In [10]:
print(num_tokens)
num_tokens=192

185


In [11]:
tokenized = []
for caption in caption_data:
    seq = []
    for i in range(num_tokens):
        try:
            seq.append(chars_arr == caption[i])
        except IndexError:
            seq.append(chars_arr == '\0')
    tokenized.append(seq)

In [19]:
print(len(chars_arr))

72


In [12]:
tokenized_captions = np.array(tokenized)

In [13]:
caption_matching_images = []
for id in caption_data_image_id:
    caption_matching_images.append(image_data[image_data_id==id])
caption_matching_images = np.array(caption_matching_images)

In [14]:
np.where(chars_arr == '\0')

(array([54], dtype=int64),)

In [15]:
print(tokenized_captions.shape)

(5004, 192, 72)


In [16]:
mask =  tf.where(tokenized_captions[:, :, np.where(chars_arr == '\0')], -1e9, 0.0) 

In [92]:
import gc
gc.collect()

0

In [22]:
A = DensePCNLayer(128, 0.00001)
C = DensePCNLayer(200, 0.00001)
A.state = tf.Variable(tf.random.normal((1, 192, 128)))
B = TransformerPCNLayer(3, 128, 8, 0.00001, A, [C], mask[0][None])
A.next_layers=[B]
C.prev_layer = B
b_out = B(A.state)
c_out = C(b_out)

In [23]:
for i in range(10):
    for layer in B.get_layers():
        layer.update_state()
        layer.update_wts()
        layer.update_b()
    print(tf.reduce_sum((B.predict_next() - B(A.predict_next()))**2)
        +tf.reduce_sum((C.predict_next() - C(B.predict_next()))**2)
        +tf.reduce_sum((A.predict_next()-B.predict_prev())**2)
        +tf.reduce_sum((B.predict_next()-C.predict_prev())**2))

tf.Tensor(225544.53, shape=(), dtype=float32)
tf.Tensor(222998.22, shape=(), dtype=float32)
tf.Tensor(220503.75, shape=(), dtype=float32)
tf.Tensor(218059.17, shape=(), dtype=float32)
tf.Tensor(215662.56, shape=(), dtype=float32)
tf.Tensor(213312.34, shape=(), dtype=float32)
tf.Tensor(211006.98, shape=(), dtype=float32)
tf.Tensor(208745.19, shape=(), dtype=float32)
tf.Tensor(206525.73, shape=(), dtype=float32)
tf.Tensor(204347.45, shape=(), dtype=float32)


In [28]:
print(tokenized_captions.shape)

(5004, 192, 72)


In [94]:
from encoder_encoder_pcn import EncoderEncoderPCN
eepcn = EncoderEncoderPCN(1e-5)
eepcn.pass_through(tf.cast(caption_matching_images[0], tf.float32), tf.cast(tf.convert_to_tensor(tokenized_captions[0][None]), tf.float32) )

ResourceExhaustedError: {{function_node __wrapped__RandomStandardNormal_device_/job:localhost/replica:0/task:0/device:CPU:0}} OOM when allocating tensor with shape[921600,307200] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator cpu [Op:RandomStandardNormal]

In [38]:
(30*30*1024)/(3*4096)

75.0

In [41]:
tf.sqrt(75.)

<tf.Tensor: shape=(), dtype=float32, numpy=8.6602545>

In [44]:
tf.sqrt(8.66)

<tf.Tensor: shape=(), dtype=float32, numpy=2.942788>

In [61]:
3*4096*3.6

44236.8

In [64]:
136*136*256/(12*2048)

192.66666666666666

In [65]:
192.66666666666666**.5

13.880441875771343

In [66]:
13.880441875771343**.5

3.725646504403141

In [68]:
136*136*256/3.7/3.7

345871.1468224981

In [71]:
(12*2048)*3.7*3.7

336445.44000000006

In [80]:
(568*568*64)/(192*512)

210.04166666666666

In [81]:
210.04166666666666**.5

14.492814311467138

In [82]:
14.492814311467138**.5

3.8069429088793987

In [84]:
(568*568*64)/3.8/3.8

1429912.4653739613

In [79]:
48*1024*3.78*3.78

702303.4367999999

In [88]:
(192*512)*3.8

373555.19999999995

In [ ]:
# Measure parameter + stored-activation memory for the current `EncoderEncoderPCN` instance.
import tensorflow as tf
from encoder_encoder_pcn import EncoderEncoderPCN

# instantiate model and run one forward pass to initialize layer states/weights
eepcn = EncoderEncoderPCN(1e-5)
try:
    eepcn.pass_through(tf.cast(caption_matching_images[0], tf.float32), tf.cast(tf.convert_to_tensor(tokenized_captions[0][None]), tf.float32))
except Exception as e:
    print('pass_through raised:', e)


def count_elements(t):
    try:
        return int(tf.size(t).numpy())
    except Exception:
        return 0

# Parameters
total_param_elems = 0
for layer in eepcn.trainable_layers:
    for name in ('wts','b','gamma','beta'):
        if hasattr(layer, name):
            v = getattr(layer, name)
            if v is not None:
                total_param_elems += count_elements(v)

# Stored states / activations
total_state_elems = 0
for layer in eepcn.trainable_layers:
    if hasattr(layer, 'state') and getattr(layer, 'state') is not None:
        total_state_elems += count_elements(layer.state)
    elif hasattr(layer, 'output_shape') and getattr(layer, 'output_shape') is not None:
        shp = getattr(layer, 'output_shape')
        try:
            prod = 1
            for s in shp:
                prod *= int(s)
            total_state_elems += prod
        except Exception:
            pass

# bytes (assume float32)
param_mb = total_param_elems * 4 / (1024**2)
state_mb = total_state_elems * 4 / (1024**2)

print(f"Parameter elements: {total_param_elems:,}  ≈ {param_mb:.1f} MB (float32)")
print(f"Stored activation elements: {total_state_elems:,}  ≈ {state_mb:.1f} MB (float32)")
print(f"Estimated total (params + stored activations): {(param_mb+state_mb):.1f} MB")

print('\nNotes:')
print('- Peak memory will be higher due to intermediates, gradients, and TF overhead.')
print('- If this cell errors with ModuleNotFoundError, run the notebook kernel that has TensorFlow installed.')